# Earth Engine Landslide Risk Analysis

This notebook demonstrates how to:
1. Setup Earth Engine Python API with service account authentication
2. Load population and elevation data for Costa Rica
3. Calculate landslide risk (Population × Slope)
4. Visualize results interactively
5. Export risk map to Google Cloud Storage as Cloud Optimized GeoTIFF

## 1. Install Dependencies

In [ ]:
!pip install earthengine-api geemap

## 2. Import Libraries

In [ ]:
import ee
import geemap
import json
import os

## 3. Authenticate with Earth Engine

**IMPORTANT:** Store your service account credentials securely:
- In Google Colab: Use Colab Secrets (left sidebar)
- In production: Use environment variables or GCP Secret Manager
- Never commit credentials to git repositories

Your service account JSON should have these keys:
- `type`, `project_id`, `private_key_id`, `private_key`
- `client_email`, `client_id`, `auth_uri`, `token_uri`
- `auth_provider_x509_cert_url`, `client_x509_cert_url`

In [ ]:
# Load service account credentials from environment or Colab secrets
# Option 1: From environment variable
try:
    SERVICE_ACCOUNT_KEY_JSON = json.loads(os.environ['EE_SERVICE_ACCOUNT_KEY'])
except KeyError:
    print("⚠️ EE_SERVICE_ACCOUNT_KEY not found in environment.")
    print("Please set your service account JSON as an environment variable or use Colab secrets.")
    # Option 2: For Colab, use userdata module (uncomment if using Colab)
    # from google.colab import userdata
    # SERVICE_ACCOUNT_KEY_JSON = json.loads(userdata.get('EE_SERVICE_ACCOUNT_KEY'))
    raise

# Extract credentials
EE_SERVICE_ACCOUNT_EMAIL = SERVICE_ACCOUNT_KEY_JSON['client_email']
EE_SERVICE_ACCOUNT_KEY_DATA = SERVICE_ACCOUNT_KEY_JSON['private_key']
EE_PROJECT_ID = SERVICE_ACCOUNT_KEY_JSON['project_id']

# Initialize Earth Engine
credentials = ee.ServiceAccountCredentials(
    EE_SERVICE_ACCOUNT_EMAIL, key_data=EE_SERVICE_ACCOUNT_KEY_DATA)
ee.Initialize(credentials, project=EE_PROJECT_ID)

print(f"✅ Earth Engine initialized with project: {EE_PROJECT_ID}")

## 4. Test Earth Engine API

Quick test to verify authentication works.

In [ ]:
# Test API by printing Mount Everest's elevation
dem = ee.Image('USGS/SRTMGL1_003')
xy = ee.Geometry.Point([86.9250, 27.9881])
elev = dem.sample(xy, 30).first().get('elevation').getInfo()
print(f'Mount Everest elevation: {elev} meters')

## 5. Define Area of Interest (AOI)

Focus on San José and Escazú area in Costa Rica.

In [ ]:
# Define AOI - San José and Escazú area (5km buffer)
aoi_center = [-84.15911, 9.93404]  # Longitude, Latitude
aoi = ee.Geometry.Point(aoi_center).buffer(5000)  # 5km buffer
aoi = ee.Geometry(aoi)  # Ensure it's a Geometry object

print(f"✅ AOI defined: {aoi_center} with 5km buffer")

## 6. Load Population Data (Severity Factor)

Population density represents the **severity** of risk - how many people would be affected if a landslide occurs.

In [ ]:
# Load WorldPop 2020 population data for Costa Rica
dataset = ee.ImageCollection('WorldPop/POP')
filtered = dataset.filter(
    ee.Filter.And(
        ee.Filter.eq('year', 2020),
        ee.Filter.eq('country', 'CRI')  # Costa Rica
    )
)

# Check if data exists
collection_size = filtered.size().getInfo()
print(f"Found {collection_size} WorldPop images for Costa Rica 2020")

if collection_size == 0:
    raise ValueError("No WorldPop data found for Costa Rica 2020")

# Get population image and clip to AOI
population_image = filtered.first().select('population')
pop_cr = population_image.clip(aoi).unmask(0)

# Normalize population for risk calculation (0 to 1)
# Formula: min(population / 200, 1)
# 200 people/km² = threshold for "high density"
max_pop_threshold = 200
normalized_pop = pop_cr.divide(max_pop_threshold).min(1).rename('normalized_population')

print(f"✅ Population data loaded and normalized (threshold: {max_pop_threshold} people/km²)")

## 7. Load Elevation and Calculate Slope (Probability Factor)

Slope represents the **probability** of landslide occurrence - steeper slopes are more susceptible.

In [ ]:
# Load NASA DEM elevation data
elevation = ee.Image('NASA/NASADEM_HGT/001').select('elevation').clip(aoi)

# Calculate slope in degrees
slope_image = ee.Terrain.slope(elevation)

# Normalize slope for risk calculation (0 to 1)
# Formula: min(slope / 45°, 1)
# 45° = threshold for "very steep" terrain
max_slope_threshold = 45
landslide_probability = slope_image.divide(max_slope_threshold).min(1).rename('landslide_probability')

print(f"✅ Elevation and slope calculated (threshold: {max_slope_threshold}°)")

## 8. Calculate Total Risk

**Risk = Severity × Probability**

- Severity: Normalized population density (0-1)
- Probability: Normalized slope (0-1)
- Risk: Product of both factors (0-1)

This formula ensures:
- No people = No risk (even on steep slopes)
- Flat terrain = No risk (even with high population)
- Highest risk where steep slopes meet dense populations

In [ ]:
# Calculate risk image
risk_image = normalized_pop.multiply(landslide_probability).rename('total_risk')

print("✅ Risk calculated: RISK = SEVERITY (population) × PROBABILITY (slope)")

## 9. Visualize Results Interactively

In [ ]:
# Create interactive map
Map = geemap.Map(center=[aoi_center[1], aoi_center[0]], zoom=12)
Map.setOptions('HYBRID')

# Add AOI boundary
Map.add_layer(aoi, {'color': 'cyan', 'fillColor': '00000000'}, 'AOI Boundary', True)

# Visualization parameters
pop_vis = {
    'bands': ['population'],
    'min': 0.0,
    'max': 200.0,
    'palette': ['24126c', '1fff4f', 'd4ff50']
}

slope_vis = {
    'min': 0,
    'max': 1,
    'palette': ['green', 'yellow', 'red']
}

risk_vis = {
    'min': 0.0,
    'max': 1.0,
    'palette': ['white', 'blue', 'orange', 'red', 'darkred']
}

# Add layers
Map.addLayer(pop_cr, pop_vis, 'Population Density (Severity)', False)
Map.addLayer(landslide_probability, slope_vis, 'Landslide Probability (Slope)', False)
Map.addLayer(risk_image, risk_vis, 'Total Risk (Pop × Slope)', True)

print("✅ Map created with 3 layers")
Map

## 10. Export Risk Map to Google Cloud Storage

Export the risk visualization as a Cloud Optimized GeoTIFF (COG) for web serving.

In [ ]:
# Define your GCS bucket name
GCS_BUCKET = 'macho-raster'  # Change this to your bucket
FILE_PREFIX = 'risk_layers/cr_2020'

# Create RGB visualization of risk
risk_viz = risk_image.visualize(
    min=0, 
    max=1, 
    palette=['white', 'blue', 'orange', 'red', 'darkred']
)

# Export task
task = ee.batch.Export.image.toCloudStorage(
    image=risk_viz,
    description='landslide_risk_cr_2020',
    bucket=GCS_BUCKET,
    fileNamePrefix=FILE_PREFIX,
    region=aoi,
    scale=30,  # 30m resolution
    crs='EPSG:3857',  # Web Mercator for web maps
    maxPixels=1e13,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True}
)

task.start()
print(f"✅ Export task started: {task.id}")
print(f"   Destination: gs://{GCS_BUCKET}/{FILE_PREFIX}.tif")
print("   Check status with: task.status()")

In [ ]:
# Check export status
print(task.status())

## Summary

This notebook demonstrated:

1. ✅ **Authentication**: Service account setup with Earth Engine
2. ✅ **Data Loading**: WorldPop 2020 population and NASA DEM elevation
3. ✅ **Risk Calculation**: Multiplication of normalized population × slope
4. ✅ **Visualization**: Interactive map with geemap
5. ✅ **Export**: Cloud Optimized GeoTIFF to GCS for web serving

### Risk Formula
```
SEVERITY = min(population / 200, 1)
PROBABILITY = min(slope° / 45°, 1)
RISK = SEVERITY × PROBABILITY
```

### Why River Valleys Show Highest Risk
- **High Population**: Water access, fertile land, transportation routes
- **Steep Slopes**: V-shaped valleys with 30-45° terrain
- **Result**: Maximum risk scores (0.7-1.0) in river valley settlements

### Next Steps
- Serve the exported GeoTIFF with TiTiler
- Create interactive web map with MapLibre GL JS
- Validate results with historical landslide data